# Demo Regression UCI

A GPflow 2 UCI-regression workflow using `demos/datasets.py`.
Install the demo extra before running this notebook; missing CSV files are downloaded by the dataset helper.


In [ ]:
import os
import sys
import numpy as np
import gpflow
from gpflow.likelihoods import Gaussian
from gpflow.optimizers import Scipy
from gpflow.utilities import set_trainable

HERE = os.getcwd()
DEMOS = HERE if os.path.basename(HERE) == 'demos' else os.path.join(HERE, 'demos')
if DEMOS not in sys.path:
    sys.path.insert(0, DEMOS)

from datasets import Datasets
from doubly_stochastic_dgp.dgp import DGP

gpflow.config.set_default_float(np.float64)

data = Datasets(data_path='./data').all_datasets['boston'].get_data(split=0)
X, Y, Xs, Ys = [data[name] for name in ['X', 'Y', 'Xs', 'Ys']]
X, Y, Xs, Ys = X[:128], Y[:128], Xs[:64], Ys[:64]
Z = X[np.linspace(0, X.shape[0] - 1, 16, dtype=int)].copy()
kernels = [gpflow.kernels.SquaredExponential(), gpflow.kernels.SquaredExponential()]

model = DGP(X, Y, Z, kernels, Gaussian(), num_samples=2, minibatch_size=32)
model.likelihood.likelihood.variance.assign(0.05)
set_trainable(model.likelihood.likelihood.variance, False)

loss_before = model.training_loss().numpy()
Scipy().minimize(model.training_loss, model.trainable_variables, options={"maxiter": 2})
mean, var = model.predict_y(Xs, 2)
rmse = np.sqrt(np.mean((Ys - np.mean(mean.numpy(), axis=0)) ** 2))

float(loss_before), float(rmse), mean.shape, var.shape
